In [3]:
# Sys path
from sys import path
from pathlib import Path

module_path = str(Path.cwd().parents[1])

if module_path not in path:
    path.append(module_path)
    
path.append(module_path + '\\functions')


# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_rows', 1000)

import data_preparation
from classification import LR, KNN, RF, XGB

from sklearn.metrics import f1_score, roc_auc_score  

from sklearn.metrics import plot_confusion_matrix

from sklearn.decomposition import PCA

import time
import pickle

# Dataset

In [44]:
df, X, y = data_preparation.load_dataset(module_path + '\\dataset\\single_attack_FDI_SLC.csv')

# Data Analysis

In [7]:
num = y.value_counts()
num =list(np.array(num))
names = ['FD', 'SLC']

# ML models

### 1. All features

In [32]:
X_train, X_test, y_train, y_test = data_preparation.split_data(X, y, test_size=0.2, random_state=123)

train = 0

if train==1:
    # training
    train_LR = -time.time()
    LR(X_train, y_train, normalize=False, save_model='\\SE_identification\\classification_models_1\\lr_model.pickle', save_param='\\SE_identification\\classification_models_1\\lr_parametres.pickle')
    train_LR += time.time()
    print('LR')

    train_KNN = -time.time()
    KNN(X_train, y_train, normalize=False, save_model='\\SE_identification\\classification_models_1\\knn_model.pickle', save_param='\\SE_identification\\classification_models_1\\knn_parametres.pickle')
    train_KNN += time.time()
    print('KNN')

    train_RF = -time.time()
    RF(X_train, y_train, normalize=False, save_model='\\SE_identification\\classification_models_1\\rf_model.pickle', save_param='\\SE_identification\\classification_models_1\\rf_parametres.pickle')
    train_RF += time.time()
    print('RF')
    
    train_XGB = -time.time()
    XGB(X_train, y_train, normalize=False, save_model='\\SE_identification\\classification_models_1\\xgb_model.pickle', save_param='\\SE_identification\\classification_models_1\\xgb_parametres.pickle', 
        save_encoder='\\SE_identification\\classification_models_1\\xgb_label_encoder.pickle')
    train_XGB += time.time()
    print('XGB')
    
    train_time = data_preparation.to_dict(train_LR, train_KNN, train_RF, train_XGB)
    data_preparation.save_model(train_time, 'time_1/train_time')
    
train_time = data_preparation.load_model('time_1/train_time')
M1_score = data_preparation.load_model('score_1/macro_score')

    
# load models
lr = pickle.load(open(module_path + '\\single_attack\\SE_identification\\classification_models_1\\lr_model.pickle', 'rb'))
knn = pickle.load(open(module_path + '\\single_attack\\SE_identification\\classification_models_1\\knn_model.pickle', 'rb'))
rf = pickle.load(open(module_path + '\\single_attack\\SE_identification\\classification_models_1\\rf_model.pickle', 'rb'))
xgb = pickle.load(open(module_path + '\\single_attack\\SE_identification\\classification_models_1\\xgb_model.pickle', 'rb'))

# 加载 encoder
le = pickle.load(open(module_path + '\\single_attack\\SE_identification\\classification_models_1\\xgb_label_encoder.pickle', 'rb'))

# load parameters
lr_param = pickle.load(open(module_path + '\\single_attack\\SE_identification\\classification_models_1\\lr_parametres.pickle', 'rb'))
knn_param = pickle.load(open(module_path + '\\single_attack\\SE_identification\\classification_models_1\\knn_parametres.pickle', 'rb'))
rf_param = pickle.load(open(module_path + '\\single_attack\\SE_identification\\classification_models_1\\rf_parametres.pickle', 'rb'))
xgb_param = pickle.load(open(module_path + '\\single_attack\\SE_identification\\classification_models_1\\xgb_parametres.pickle', 'rb'))
    
# prediction
test_LR = -time.time()
y_pred_lr = lr.predict(X_test)
test_LR += time.time()
    
test_KNN = -time.time()
y_pred_knn = knn.predict(X_test)
test_KNN += time.time()
    
test_RF = -time.time()
y_pred_rf = rf.predict(X_test)
test_RF += time.time()
    
test_XGB = -time.time()
y_pred_xgb = xgb.predict(X_test)
test_XGB += time.time()

# 转换为原始标签
y_pred_xgb = le.inverse_transform(y_pred_xgb)

# F1 score
macro_f1_lr = f1_score(y_test, y_pred_lr)
macro_f1_knn = f1_score(y_test, y_pred_knn)
macro_f1_rf = f1_score(y_test, y_pred_rf)
macro_f1_xgb = f1_score(y_test, y_pred_xgb)

In [36]:
print('##############')
print('F1 score:')
print('LR', M1_score['LR']) 
print('KNN', M1_score['KNN']) 
print('RF', M1_score['RF']) 
print('XGB', M1_score['XGB']) 

print('##############')
print('Training Time:')
print('LR', train_time['LR'], 'sec')
print('KNN', train_time['KNN'], 'sec')
print('RF', train_time['RF'], 'sec')
print('XGB', train_time['XGB'], 'sec')

print('##############')
print('Testing Time:')
print('LR', test_LR, 'sec')
print('KNN', test_KNN, 'sec')
print('RF', test_RF, 'sec')
print('XGB', test_XGB, 'sec')

print('##############')
print('Parameters:')
print('LR:', lr_param)
print('KNN:', knn_param)
print('RF:', rf_param)
print('XGB:', xgb_param)

##############
F1 score:
LR 70.9105560032232
KNN 95.2874618392054
RF 96.38174529069246
XGB 98.1239476503812
##############
Training Time:
LR 188.06270909309387 sec
KNN 11.274773120880127 sec
RF 419.46393370628357 sec
XGB 221.31778001785278 sec
##############
Testing Time:
LR 0.0050013065338134766 sec
KNN 0.07301616668701172 sec
RF 0.15157771110534668 sec
XGB 0.01900315284729004 sec
##############
Parameters:
LR: ['lbfgs', 'none', 9.78620480398541]
KNN: [6, 'distance']
RF: [734, 8, 3, 6]
XGB: [773, 8, 0.014228041454890745, 0.6145211656257424, 0.5200466602245554]


-------------------

### PCA

In [42]:
import os
import time
import pickle
import numpy as np

from sklearn.decomposition import PCA
from sklearn.metrics import f1_score


X_train_raw, X_test_raw, y_train, y_test = (
    data_preparation.split_data(
        X,
        y,
        test_size=0.2,
        random_state=123
    )
)

y_train = np.asarray(y_train).ravel()
y_test = np.asarray(y_test).ravel()

pca = PCA(
    n_components=0.95
)

X_train = pca.fit_transform(
    X_train_raw
)

X_test = pca.transform(
    X_test_raw
)

model_dir = os.path.join(
    module_path,
    'single_attack',
    'SE_identification',
    'classification_models'
)

os.makedirs(
    model_dir,
    exist_ok=True
)


pca_path = os.path.join(
    model_dir,
    'pca_model.pickle'
)

with open(
    pca_path,
    'wb'
) as f:
    pickle.dump(
        pca,
        f
    )

train = 1


if train == 1:


    # --------------------------------------------------------
    # LR
    # --------------------------------------------------------

    train_LR = -time.time()

    LR(
        X_train,
        y_train,
        normalize=False,
        save_model=(
            '\\SE_identification'
            '\\classification_models'
            '\\lr_model_pca.pickle'
        ),
        save_param=(
            '\\SE_identification'
            '\\classification_models'
            '\\lr_parametres_pca.pickle'
        )
    )

    train_LR += time.time()

    print('LR')


    # --------------------------------------------------------
    # KNN
    # --------------------------------------------------------

    train_KNN = -time.time()

    KNN(
        X_train,
        y_train,
        normalize=False,
        save_model=(
            '\\SE_identification'
            '\\classification_models'
            '\\knn_model_pca.pickle'
        ),
        save_param=(
            '\\SE_identification'
            '\\classification_models'
            '\\knn_parametres_pca.pickle'
        )
    )

    train_KNN += time.time()

    print('KNN')


    # --------------------------------------------------------
    # RF
    # --------------------------------------------------------

    train_RF = -time.time()

    RF(
        X_train,
        y_train,
        normalize=False,
        save_model=(
            '\\SE_identification'
            '\\classification_models'
            '\\rf_model_pca.pickle'
        ),
        save_param=(
            '\\SE_identification'
            '\\classification_models'
            '\\rf_parametres_pca.pickle'
        )
    )

    train_RF += time.time()

    print('RF')


    # --------------------------------------------------------
    # XGB
    # --------------------------------------------------------

    train_XGB = -time.time()

    XGB(
        X_train,
        y_train,
        normalize=False,
        save_model=(
            '\\SE_identification'
            '\\classification_models'
            '\\xgb_model_pca.pickle'
        ),
        save_param=(
            '\\SE_identification'
            '\\classification_models'
            '\\xgb_parametres_pca.pickle'
        ),
        save_encoder=(
            '\\SE_identification'
            '\\classification_models'
            '\\xgb_label_encoder_pca.pickle'
        )
    )

    train_XGB += time.time()

    print('XGB')

    train_time = data_preparation.to_dict(
        train_LR,
        train_KNN,
        train_RF,
        train_XGB
    )


    data_preparation.save_model(
        train_time,
        'time_1/train_time_pca'
    )


train_time = data_preparation.load_model(
    'time_1/train_time_pca'
)


lr = pickle.load(
    open(
        os.path.join(
            model_dir,
            'lr_model_pca.pickle'
        ),
        'rb'
    )
)


knn = pickle.load(
    open(
        os.path.join(
            model_dir,
            'knn_model_pca.pickle'
        ),
        'rb'
    )
)


rf = pickle.load(
    open(
        os.path.join(
            model_dir,
            'rf_model_pca.pickle'
        ),
        'rb'
    )
)


xgb = pickle.load(
    open(
        os.path.join(
            model_dir,
            'xgb_model_pca.pickle'
        ),
        'rb'
    )
)


le = pickle.load(
    open(
        os.path.join(
            model_dir,
            'xgb_label_encoder_pca.pickle'
        ),
        'rb'
    )
)


lr_param = pickle.load(
    open(
        os.path.join(
            model_dir,
            'lr_parametres_pca.pickle'
        ),
        'rb'
    )
)


knn_param = pickle.load(
    open(
        os.path.join(
            model_dir,
            'knn_parametres_pca.pickle'
        ),
        'rb'
    )
)


rf_param = pickle.load(
    open(
        os.path.join(
            model_dir,
            'rf_parametres_pca.pickle'
        ),
        'rb'
    )
)


xgb_param = pickle.load(
    open(
        os.path.join(
            model_dir,
            'xgb_parametres_pca.pickle'
        ),
        'rb'
    )
)

M1_score = data_preparation.load_model('score_1/macro_score_pca')


test_LR = -time.time()

y_pred_lr = lr.predict(
    X_test
)

test_LR += time.time()


test_KNN = -time.time()

y_pred_knn = knn.predict(
    X_test
)

test_KNN += time.time()


test_RF = -time.time()

y_pred_rf = rf.predict(
    X_test
)

test_RF += time.time()


test_XGB = -time.time()

y_pred_xgb = xgb.predict(
    X_test
)

test_XGB += time.time()


# XGB label restore

y_pred_xgb = le.inverse_transform(
    np.asarray(y_pred_xgb)
    .ravel()
    .astype(int)
)



macro_f1_lr = f1_score(
    y_test,
    y_pred_lr,
    average='macro'
)


macro_f1_knn = f1_score(
    y_test,
    y_pred_knn,
    average='macro'
)


macro_f1_rf = f1_score(
    y_test,
    y_pred_rf,
    average='macro'
)


macro_f1_xgb = f1_score(
    y_test,
    y_pred_xgb,
    average='macro'
)


print('###############')

print('F1 score:')

print('LR', M1_score['LR']) 
print('KNN', M1_score['KNN']) 
print('RF', M1_score['RF']) 
print('XGB', M1_score['XGB']) 

print('###############')

print('Training Time:')

print('LR', train_time['LR'], 'sec')
print('KNN', train_time['KNN'], 'sec')
print('RF', train_time['RF'], 'sec')
print('XGB', train_time['XGB'], 'sec')


print('###############')

print('Testing Time:')

print('LR', test_LR, 'sec')
print('KNN', test_KNN, 'sec')
print('RF', test_RF, 'sec')
print('XGB', test_XGB, 'sec')


print('###############')

print('Parameters:')

print('LR:', lr_param)
print('KNN:', knn_param)
print('RF:', rf_param)
print('XGB:', xgb_param)

LR
KNN
RF
XGB
###############
F1 score:
LR 72.39186527481367
KNN 93.74261859370425
RF 95.79638152490713
XGB 96.60372829173829
###############
Training Time:
LR 4.120627403259277 sec
KNN 2.9437408447265625 sec
RF 63.059659004211426 sec
XGB 20.721395254135132 sec
###############
Testing Time:
LR 0.0010001659393310547 sec
KNN 0.12002825736999512 sec
RF 0.21293377876281738 sec
XGB 0.01600050926208496 sec
###############
Parameters:
LR: ['newton-cg', 'none', 6.028030997340369]
KNN: [29, 'uniform']
RF: [966, 6, 3, 6]
XGB: [438, 12, 0.02237947889232047, 0.7647517763082216, 0.9345531258126619]
